# 📘 Day 12 — Multi-Layer Perceptrons (MLP)
> **Ashraf | Indian Institute of Technology (ISM), Dhanbad**  
> Chapter 4.2 — Deep Learning
---

## 🧠 Theory 1 — From Perceptron to Multi-Layer Perceptron

A **single perceptron** can only learn a linear boundary. Real-world data is rarely linearly separable.

A **Multi-Layer Perceptron (MLP)** solves this by stacking multiple layers of neurons, each applying a non-linear activation — allowing the network to model **any function** (universal approximator).

### Architecture

```
Input Layer     Hidden Layer(s)     Output Layer
   x₁ ──┐
   x₂ ──┼──► [h₁ h₂ h₃ h₄ h₅] ──► [o₁ o₂ o₃]
   x₃ ──┤        (non-linear)         (output)
   x₄ ──┘
```

### Output of a Fully Connected (Dense) Layer

$$
\mathbf{h} = f\left(\mathbf{W}\mathbf{x} + \mathbf{b}\right)
$$

| Symbol | Meaning |
|--------|---------|
| $\mathbf{x}$ | Input feature vector |
| $\mathbf{W}$ | Weight matrix (learned) |
| $\mathbf{b}$ | Bias vector (learned) |
| $f(\cdot)$ | Activation function |
| $\mathbf{h}$ | Output (hidden or final) |

> 💡 **Dense/Fully Connected**: every neuron in layer $l$ connects to every neuron in layer $l+1$.

## ⚡ Theory 2 — Activation Functions

Activation functions introduce **non-linearity** so the network can learn complex patterns.

| Function | Formula | Range | Best Used For |
|----------|---------|-------|---------------|
| **ReLU** | $f(x) = \max(0, x)$ | $[0, \infty)$ | Hidden layers (most popular) |
| **Sigmoid** | $f(x) = \dfrac{1}{1+e^{-x}}$ | $(0, 1)$ | Binary classification output |
| **Tanh** | $f(x) = \tanh(x)$ | $(-1, 1)$ | Hidden layers, zero-centred |

### Why ReLU is Popular
- Derivatives are **simple**: either 0 or 1
- Avoids the **vanishing gradient** problem
- Fast to compute
- Sparse activation (not all neurons fire)

> ⚠️ **Vanishing gradients**: Sigmoid/Tanh saturate at extremes → gradients → 0 → learning stops in deep networks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

x = np.linspace(-4, 4, 200)

def sigm(x): return 1 / (1 + np.exp(-x))
def relu(x): return np.maximum(x, 0)
tanh = np.tanh(x)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
configs = [
    (sigm(x), 'Sigmoid', '#e74c3c', r'$\frac{1}{1+e^{-x}}$'),
    (relu(x), 'ReLU',    '#2980b9', r'$\max(0, x)$'),
    (tanh,    'Tanh',    '#27ae60', r'$\tanh(x)$'),
]
for ax, (y, name, col, formula) in zip(axes, configs):
    ax.plot(x, y, color=col, linewidth=2.5)
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_title(f'{name}\n{formula}', fontsize=12)
    ax.set_xlabel('x')
    ax.grid(alpha=0.3, linestyle='dashed')
    ax.set_xlim([-4, 4])

fig.suptitle('Activation Functions Comparison', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## 🏗️ Theory 3 — Typical MLP Structures

### Regression MLP
- **Output**: one or more real-valued scalars
- **Output activation**: None (identity), ReLU (positive values), Sigmoid/Tanh (bounded)
- **Loss function**: MSE, MAE

```
Input → [Dense+ReLU] → [Dense+ReLU] → [Dense] → scalar output
```

### Classification MLP
- **Binary**: 1 output neuron + Sigmoid → probability ∈ (0,1)
- **Multi-class**: $n$ output neurons + Softmax → probability distribution over $n$ classes
- **Loss function**: Cross-Entropy

```
Input → [Dense+ReLU] → [Dense+ReLU] → [Dense+Softmax] → class probs
```

| Task | Output neurons | Output activation | Loss |
|------|---------------|-------------------|------|
| Regression | = # values | None / ReLU | MSE |
| Binary classification | 1 | Sigmoid | Binary cross-entropy |
| Multi-class | = # classes | Sigmoid / Softmax | Cross-entropy |

## 🔄 Theory 4 — Training Neural Networks

Training a neural network involves two key passes:

### 1. Forward Pass
Data flows **input → output**, computing predictions layer by layer:
$$
\mathbf{h}^{(l)} = f\left(\mathbf{W}^{(l)}\mathbf{h}^{(l-1)} + \mathbf{b}^{(l)}\right)
$$

### 2. Loss Computation
Compare prediction $\hat{y}$ to true label $y$ using a **loss function** (e.g., cross-entropy):
$$
\mathcal{L} = -\sum_c y_c \log(\hat{y}_c)
$$

### 3. Backpropagation
Gradients are computed **output → input** using the **chain rule**:
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{W}^{(l)}} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}^{(l)}} \cdot \frac{\partial \mathbf{h}^{(l)}}{\partial \mathbf{W}^{(l)}}
$$

### 4. Weight Update (Gradient Descent)
$$
\mathbf{W} \leftarrow \mathbf{W} - \eta \cdot \frac{\partial \mathcal{L}}{\partial \mathbf{W}}
$$

### Training Loop

```
For each epoch:
  For each mini-batch:
    1. Forward pass  → compute ŷ
    2. Compute loss  → L(y, ŷ)
    3. Backward pass → ∂L/∂W via chain rule
    4. Update weights → W = W - η·∇W
```

> 💾 Backprop requires storing **all intermediate activations** in memory — more RAM than inference!

## 🎲 Theory 5 — Dropout (Regularisation)

**Dropout** is a regularisation technique that randomly **zeros out** a fraction of neurons during training.

$$
\tilde{h}_i = h_i \cdot \text{Bernoulli}(1 - p)
$$

| Aspect | Detail |
|--------|--------|
| **During training** | Each neuron dropped with probability $p$ |
| **During inference** | All neurons active (weights scaled by $1-p$) |
| **Effect** | Prevents co-adaptation; acts like an ensemble |
| **Typical $p$** | 0.2–0.5 |

```
Normal layer:  [h₁  h₂  h₃  h₄  h₅]
After dropout: [h₁   0  h₃   0  h₅]  ← random zeros
```

> 💡 Dropout forces the network to learn **redundant representations** — no single neuron can be relied upon.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)

## 📦 Data Preparation — Digits Dataset

We use sklearn's **digits dataset**: 1797 images of handwritten digits (0–9), each 8×8 pixels.

```
Image (8×8 grid)    Flattened (64 values)    Label
┌────────────┐
│ 0 0 5 13..│  →   [0, 0, 5, 13, ...]   →   5
└────────────┘
```

We split into **80% train / 20% test** and wrap in PyTorch `DataLoader` for mini-batch training.

In [ ]:
from sklearn.datasets import load_digits
from torch.utils.data import Dataset, DataLoader, random_split

digits = load_digits()
data, labels = digits["data"].copy(), digits["target"].copy()
data = data.astype(np.float32).reshape(-1, 8, 8)

print(f"Data shape  : {data.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Classes     : {np.unique(labels)}")

In [ ]:
# Visualise a few sample digits
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for ax, img, lbl in zip(axes.ravel(), data[:10], labels[:10]):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(f'Label: {lbl}', fontsize=11)
    ax.axis('off')
fig.suptitle('Sample Digits from Dataset', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data   = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return torch.Tensor(self.data[index]), self.labels[index]

custom_dataset = CustomDataset(data, labels)

train_size = int(0.8 * len(custom_dataset))
test_size  = len(custom_dataset) - train_size
train_dataset, test_dataset = random_split(custom_dataset, [train_size, test_size])

batch_size = 32
data_loader_train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
data_loader_test  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print(f"Training samples : {train_size}")
print(f"Test samples     : {test_size}")
print(f"Batches per epoch: {len(data_loader_train)}")

## 🏛️ Theory 6 — MLP Architecture Design

### Our Network: 3-Layer Classification MLP

```
Input (8×8 image)
      ↓
  Flatten → 64 values
      ↓
  Dense(64→32) + ReLU    ← Hidden Layer 1
      ↓
  Dense(32→16) + ReLU    ← Hidden Layer 2
      ↓
  Dense(16→10) + Sigmoid ← Output Layer (10 digit classes)
```

### Design Choices

| Component | Choice | Reason |
|-----------|--------|--------|
| Flatten | 8×8 → 64 | MLP needs 1D input |
| Hidden layers | 2 | Balance complexity vs. overfitting |
| Hidden neurons | 32, 16 | Funnel architecture |
| Hidden activation | ReLU | Fast, no vanishing gradients |
| Output neurons | 10 | One per digit class |
| Output activation | Sigmoid | Multi-label probabilities |
| Loss | CrossEntropyLoss | Standard for multi-class |
| Optimiser | SGD | Simple, reliable |

In [ ]:
torch.manual_seed(42)

class my_mlp(nn.Module):
    def __init__(self, size_img, num_classes):
        super(my_mlp, self).__init__()
        self.flatten      = nn.Flatten()
        self.layer1       = nn.Linear(size_img * size_img, 32)
        self.relu1        = nn.ReLU()
        self.layer2       = nn.Linear(32, 16)
        self.relu2        = nn.ReLU()
        self.output_layer = nn.Linear(16, num_classes)
        self.sigmoid      = nn.Sigmoid()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        x = self.sigmoid(self.output_layer(x))
        return x

model = my_mlp(8, 10)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

## 🎛️ Theory 7 — Loss Functions & Optimisers

### Cross-Entropy Loss (Multi-class)
$$
\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)
$$
- Penalises **confident wrong predictions** heavily
- Standard for classification tasks

### Optimisers

| Optimiser | Update Rule | Pros | Cons |
|-----------|-------------|------|------|
| **SGD** | $w \leftarrow w - \eta \nabla w$ | Simple, stable | Slow, sensitive to $\eta$ |
| **Adam** | Adaptive per-parameter LR | Fast, robust | More memory |
| **RMSProp** | Divides by moving avg of gradients | Good for RNNs | Less interpretable |

### Checkpointing

Saving model state periodically during training lets you:
- **Resume** training after interruption
- **Revert** to best-performing epoch
- **Compare** snapshots

```python
checkpoint = {
    'epoch': epoch,
    'state_dict': model.state_dict(),    # weights
    'optimizer': optimizer.state_dict()  # optimiser state
}
torch.save(checkpoint, 'checkpoint.pt')
```

In [ ]:
def train(model, n_epochs, trainloader, testloader=None, learning_rate=0.001):

    dir0 = './my_mlp_checkpoint'
    os.makedirs(dir0, exist_ok=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)

    loss_time     = np.zeros(n_epochs)
    accuracy_time = np.zeros(n_epochs)

    for epoch in range(n_epochs):
        running_loss = 0
        model.train()

        for data in trainloader:
            inputs, labels_batch = data[0].float(), data[1].long()
            optimizer.zero_grad()
            outputs = model(inputs)
            loss    = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        loss_time[epoch] = running_loss / len(trainloader)

        # Save checkpoint
        torch.save({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict()
        }, f'{dir0}/checkpoint.pt')

        # Evaluate on test set
        if testloader is not None:
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for data in testloader:
                    inputs, labels_batch = data[0].float(), data[1].long()
                    outputs = model(inputs)
                    _, predicted = torch.max(outputs.data, 1)
                    total   += labels_batch.size(0)
                    correct += (predicted == labels_batch).sum().item()
            accuracy_time[epoch] = 100 * correct / total

        if (epoch + 1) % 20 == 0:
            print(f'[Epoch {epoch+1:3d}] loss: {loss_time[epoch]:.4f}'
                  + (f'  accuracy: {accuracy_time[epoch]:.2f}%' if testloader else ''))

    return (loss_time, accuracy_time) if testloader else loss_time

## 🚀 Train the Model

We train for **200 epochs** with learning rate 0.005, printing every 20 epochs.

In [ ]:
alpha = 0.005
(loss, accuracy) = train(model, 200, data_loader_train, data_loader_test, alpha)
print("\n✅ Training complete!")
print(f"Final loss    : {loss[-1]:.4f}")
print(f"Final accuracy: {accuracy[-1]:.2f}%")

## 📈 Learning Curve

Plot **loss** and **accuracy** over epochs to diagnose training behaviour.

| Pattern | Interpretation |
|---------|---------------|
| Loss decreasing, accuracy rising | ✅ Learning correctly |
| Loss plateaus early | ⚠️ Learning rate too small or model too simple |
| Loss drops then rises | ⚠️ Overfitting |
| Loss oscillates wildly | ⚠️ Learning rate too large |

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = '#e74c3c'
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Cross-Entropy Loss', color=color1, fontsize=12)
ax1.plot(np.arange(1, len(loss)+1), loss, color=color1, linewidth=2, label='Loss')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(alpha=0.3, linestyle='dashed')

ax2 = ax1.twinx()
color2 = '#2980b9'
ax2.set_ylabel('Accuracy (%)', color=color2, fontsize=12)
ax2.plot(np.arange(1, len(accuracy)+1), accuracy, color=color2,
         linewidth=2, linestyle='--', label='Accuracy')
ax2.tick_params(axis='y', labelcolor=color2)

fig.suptitle('MLP Training Curve — Loss & Accuracy', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## 🎛️ Theory 8 — Fine-Tuning Hyperparameters

### Key Hyperparameters

| Hyperparameter | Effect |
|----------------|--------|
| **Learning rate** $\eta$ | Biggest impact; try log-scale (1e-4 to 1e-1) |
| **Batch size** | Larger = stable gradients; smaller = more updates, more noise |
| **Epochs** | More = better fit (until overfitting) |
| **Hidden units** | More = more capacity (risk: overfitting) |
| **# Layers** | Deeper = higher-order features (risk: vanishing gradients) |
| **Dropout rate** | Higher = more regularisation (risk: underfitting) |
| **Optimiser** | Adam usually outperforms plain SGD |

### Search Strategies
- **Manual** — good for building intuition
- **Grid Search** — exhaustive, expensive
- **Random Search** — more efficient than grid (Bergstra & Bengio 2012)
- **Bayesian Optimisation** — smartest, learns from past trials

> 🔧 `scikit-learn` provides `RandomizedSearchCV` for systematic hyperparameter search.

## 🔬 MLP with Scikit-Learn — Effect of Regularisation ($\alpha$)

We test 5 values of the regularisation parameter $\alpha$ on 3 different datasets to visualise how regularisation affects the decision boundary.

In [ ]:
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

h      = 0.02
alphas = np.logspace(-1, 1, 5)

classifiers, names = [], []
for alpha in alphas:
    classifiers.append(make_pipeline(
        StandardScaler(),
        MLPClassifier(solver='lbfgs', alpha=alpha, random_state=1,
                      max_iter=2000, early_stopping=True,
                      hidden_layer_sizes=[100, 100])
    ))
    names.append(f"α={alpha:.2f}")

X, y = make_classification(n_features=2, n_redundant=0, n_informative=2,
                            random_state=0, n_clusters_per_class=1)
rng = np.random.RandomState(2)
X  += 2 * rng.uniform(size=X.shape)
linearly_separable = (X, y)

datasets = [
    make_moons(noise=0.3, random_state=0),
    make_circles(noise=0.2, factor=0.5, random_state=1),
    linearly_separable
]
dataset_names = ['Moons', 'Circles', 'Linear']

figure = plt.figure(figsize=(18, 10))
i = 1
cm       = plt.cm.RdBu
cm_bright = ListedColormap(['#FF4444', '#4444FF'])

for ds_idx, (X, y) in enumerate(datasets):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4)
    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    ax = plt.subplot(len(datasets), len(classifiers)+1, i)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors='k', s=20)
    ax.scatter(X_test[:, 0],  X_test[:, 1],  c=y_test,  cmap=cm_bright, alpha=0.5, s=20)
    ax.set_xlim(xx.min(), xx.max()); ax.set_ylim(yy.min(), yy.max())
    ax.set_xticks(()); ax.set_yticks(())
    ax.set_ylabel(dataset_names[ds_idx], fontsize=10)
    if ds_idx == 0: ax.set_title('Data', fontsize=10)
    i += 1

    for name, clf in zip(names, classifiers):
        ax = plt.subplot(len(datasets), len(classifiers)+1, i)
        clf.fit(X_train, y_train)
        score = clf.score(X_test, y_test)
        Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
        ax.contourf(xx, yy, Z, cmap=cm, alpha=0.8)
        ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright,
                   edgecolors='k', s=15)
        ax.scatter(X_test[:, 0],  X_test[:, 1],  c=y_test,  cmap=cm_bright,
                   alpha=0.5, edgecolors='k', s=15)
        ax.set_xlim(xx.min(), xx.max()); ax.set_ylim(yy.min(), yy.max())
        ax.set_xticks(()); ax.set_yticks(())
        if ds_idx == 0: ax.set_title(name, fontsize=9)
        ax.text(xx.max()-.3, yy.min()+.3, f'{score:.2f}',
                size=12, ha='right', color='white', fontweight='bold')
        i += 1

figure.suptitle('MLP Decision Boundaries — Effect of Regularisation α', fontsize=14, fontweight='bold')
figure.subplots_adjust(left=.04, right=.98, top=.93, hspace=0.1, wspace=0.05)
plt.show()

## 💾 Theory 9 — Saving and Restoring a Model

PyTorch models can be saved in two ways:

### Option A — Save Only Weights (`state_dict`)
```python
# Save
torch.save(model.state_dict(), 'model_weights.pt')

# Load
model = my_mlp(8, 10)          # re-create architecture first
model.load_state_dict(torch.load('model_weights.pt'))
model.eval()
```

### Option B — Save Full Checkpoint (weights + optimiser + epoch)
```python
# Save
torch.save({
    'epoch': epoch,
    'state_dict': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'loss': loss
}, 'checkpoint.pt')

# Load
checkpoint = torch.load('checkpoint.pt')
model.load_state_dict(checkpoint['state_dict'])
optimizer.load_state_dict(checkpoint['optimizer'])
start_epoch = checkpoint['epoch']
```

| Method | Use Case |
|--------|----------|
| `state_dict` only | Inference / transfer learning |
| Full checkpoint | Resume training |

In [ ]:
# Save the trained model weights
torch.save(model.state_dict(), './my_mlp_checkpoint/final_weights.pt')
print("✅ Model weights saved.")

# Reload and verify
model_loaded = my_mlp(8, 10)
model_loaded.load_state_dict(torch.load('./my_mlp_checkpoint/final_weights.pt'))
model_loaded.eval()
print("✅ Model weights loaded successfully.")

# Quick sanity check on test batch
sample_inputs, sample_labels = next(iter(data_loader_test))
with torch.no_grad():
    preds = model_loaded(sample_inputs.float())
    _, predicted = torch.max(preds, 1)
correct = (predicted == sample_labels.long()).sum().item()
print(f"Batch accuracy (loaded model): {correct}/{len(sample_labels)} = {100*correct/len(sample_labels):.1f}%")

## 📊 Evaluate — Confusion Matrix

A confusion matrix shows how well each class is predicted vs. misclassified.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for data in data_loader_test:
        inputs, labels_batch = data[0].float(), data[1].long()
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels_batch.numpy())

cm_matrix = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_matrix,
                               display_labels=np.arange(10))
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — Digit Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

overall_acc = np.trace(cm_matrix) / np.sum(cm_matrix) * 100
print(f"Overall Test Accuracy: {overall_acc:.2f}%")

## 🔑 Summary

```
┌──────────────────────────────────────────────────────────────┐
│               DAY 12 — MLP KEY TAKEAWAYS                     │
├──────────────────────────────────────────────────────────────┤
│  ✅ MLP = stacked perceptrons with non-linear activations    │
│  ✅ ReLU is the go-to hidden layer activation                │
│  ✅ Forward pass: input → prediction                         │
│  ✅ Backprop: chain rule → gradients → weight updates        │
│  ✅ Dropout prevents overfitting by random neuron zeroing    │
│  ✅ CrossEntropyLoss for classification tasks                │
│  ✅ Checkpoints save training state for recovery             │
│  ✅ Hyperparameter tuning: LR, depth, width, dropout         │
│  ✅ PyTorch: define → train → evaluate → save                │
└──────────────────────────────────────────────────────────────┘
```

> 🚀 **Next:** Day 13 — Convolutional Neural Networks (CNNs) use **spatial weight sharing** to process image data far more efficiently than MLPs.